# Class-balanced T1 training with MiniCPM-V 4.6 (MLX)

This is a separate MiniCPM-V 4.6 experiment. It exports the public T1 splits to the MLX-VLM LoRA schema (`images` plus model-formatted `messages`), balances training records only, performs an image-bearing four-example smoke run, trains once, and evaluates the saved adapter on validation and test. Test examples are not used for training. The recorded results and limits are summarised in `docs/T1_MINICPM_RESULTS.md`.

The training prompt asks the model to inspect the whole bright-field blood-smear field, look for elongated/curved Trypanosoma with an undulating outline or flagellum, ignore artifacts and filenames, and return exactly `trypanosome_present` or `negative_control`. MiniCPM-V 4.6 is loaded from the pinned `openbmb/MiniCPM-V-4.6` revision and converted locally for MLX. Visual token compression is set to **4x** for microscopy detail; the processor and model are configured consistently.

The notebook installs upstream MLX-VLM with its `minicpmv4_6` model-specific formatter and trainer support, then checks that the installed package exposes that model type before proceeding. See the [MLX-VLM LoRA guide](https://github.com/Blaizzy/mlx-vlm/blob/main/mlx_vlm/LORA.MD), [MiniCPM-V 4.6 model card](https://huggingface.co/openbmb/MiniCPM-V-4.6), and [MLX-VLM MiniCPM-V 4.6 implementation](https://github.com/Blaizzy/mlx-vlm/tree/main/mlx_vlm/models/minicpmv4_6).

In [1]:
%pip install -q 'mlx-vlm[train] @ git+https://github.com/Blaizzy/mlx-vlm.git' 'scikit-learn>=1.5'
import platform, importlib.metadata as metadata
import mlx.core as mx
from mlx_vlm.prompt_utils import MODEL_CONFIG
print('CPU:', platform.machine(), '| MLX:', metadata.version('mlx'), '| mlx-vlm:', metadata.version('mlx-vlm'))
if platform.machine() != 'arm64' or not mx.metal.is_available():
    raise RuntimeError('Use a native Apple Silicon Python/Jupyter kernel with Metal support')
if 'minicpmv4_6' not in MODEL_CONFIG:
    raise RuntimeError('Installed mlx-vlm does not include MiniCPM-V 4.6 support; restart the kernel after installing current main')

Note: you may need to restart the kernel to use updated packages.


/Users/ivywatitwa/medgemma-vet-diagnostics/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CPU: arm64 | MLX: 0.32.2 | mlx-vlm: 0.7.3


In [3]:
from pathlib import Path
import subprocess, sys, json, shutil

# Open this notebook from the repository root, or set ROOT explicitly.
ROOT = Path.cwd().resolve()
TRAINER = ROOT / 'scripts/train_t1_minicpm_mlx.py'
DATA_DIR = ROOT / 'data/secondary/mlx_t1_minicpmv46'
MODEL_ID = 'openbmb/MiniCPM-V-4.6'
MODEL_REVISION = '36f34a661a4bd35d0dc2294cb044d2584646c7d3'
MODEL_DIR = ROOT / 'checkpoints/mlx-openbmb-minicpm-v46-4bit'
OUTPUT_DIR = ROOT / 'adapters/minicpm-v46-t1-balanced-prompt-v1'
SMOKE_DIR = ROOT / 'adapters/minicpm-v46-t1-smoke-prompt-v1'
assert TRAINER.is_file() and (DATA_DIR / 'train.jsonl').is_file() and (DATA_DIR / 'val.jsonl').is_file(), f'Set ROOT to the repository and generate the MLX dataset first: {ROOT}'
print('Root:', ROOT, '| Free disk (GiB):', round(shutil.disk_usage(ROOT).free / 2**30, 1))
subprocess.run([sys.executable, '-u', str(TRAINER), '--root', str(ROOT), '--data-dir', str(DATA_DIR), '--mode', 'check'], check=True)
PROMPT = json.loads((DATA_DIR / 'dataset_info.json').read_text())['prompt']
print('Training prompt:', PROMPT)

Root: /Users/ivywatitwa/medgemma-vet-diagnostics | Free disk (GiB): 20.6
train: 4332; {'negative_control': 2166, 'trypanosome_present': 2166}; all image paths resolve
val: 488; {'negative_control': 16, 'trypanosome_present': 472}; all image paths resolve
Training prompt: Inspect the entire bright-field blood-smear microscopy image for Trypanosoma parasites. Look for an elongated, curved protozoan with an undulating membrane or flagellum among the blood cells; ignore stain artifacts, debris, and filenames. Reply with exactly one label: `trypanosome_present` if at least one Trypanosoma is visible, or `negative_control` if none is visible. Do not add an explanation.


## Convert the pinned OpenBMB checkpoint
The source model is public. Small tokenizer/config files are staged with owner-write permissions, while the large source weights are symlinked from the Hugging Face cache. This avoids a Hub-cache mode-bit issue where `tokenizer.json` is copied as read-only and Transformers cannot rewrite it at the end of conversion. If a prior conversion failed after quantizing weights, this cell repairs the partial output and retries using the cached source files. A provenance marker is written only after conversion completes.

In [8]:
from huggingface_hub import snapshot_download
import stat
SOURCE = {'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'quantization': 'mlx-4bit-affine', 'q_bits': 4, 'q_group_size': 64}
marker = MODEL_DIR / 'model_source.json'
if marker.is_file():
    if json.loads(marker.read_text()) != SOURCE:
        raise RuntimeError(f'Existing conversion provenance differs: {MODEL_DIR}')
    print('Reusing pinned conversion:', MODEL_DIR)
else:
    if MODEL_DIR.exists():
        required_partial = [MODEL_DIR / 'model.safetensors', MODEL_DIR / 'config.json']
        if not all(path.is_file() for path in required_partial):
            raise RuntimeError(f'Incomplete conversion directory; inspect/remove this generated path before retrying: {MODEL_DIR}')
        copied_tokenizer = MODEL_DIR / 'tokenizer.json'
        if copied_tokenizer.exists():
            copied_tokenizer.chmod(copied_tokenizer.stat().st_mode | stat.S_IWUSR)
        print('Repairing interrupted conversion in:', MODEL_DIR)
    else:
        MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    hf_snapshot = Path(snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION))
    HF_STAGE = MODEL_DIR.parent / 'minicpm-v46-hf-source-writable'
    if HF_STAGE.exists():
        shutil.rmtree(HF_STAGE)
    HF_STAGE.mkdir(parents=True)
    writable_suffixes = {'.json', '.jinja', '.txt', '.model', '.py'}
    for source_file in hf_snapshot.iterdir():
        if not source_file.is_file():
            continue
        staged_file = HF_STAGE / source_file.name
        if source_file.suffix in writable_suffixes:
            shutil.copyfile(source_file, staged_file)
            staged_file.chmod(staged_file.stat().st_mode | stat.S_IWUSR)
        else:
            staged_file.symlink_to(source_file.resolve())
    assert (HF_STAGE / 'config.json').is_file() and (HF_STAGE / 'tokenizer.json').is_file() and (HF_STAGE / 'model.safetensors').is_file()
    subprocess.run([sys.executable, '-m', 'mlx_vlm', 'convert', '--hf-path', str(HF_STAGE), '--mlx-path', str(MODEL_DIR), '--quantize', '--q-bits', '4', '--q-group-size', '64'], check=True)
    assert (MODEL_DIR / 'config.json').is_file() and (MODEL_DIR / 'tokenizer_config.json').is_file()
    marker.write_text(json.dumps(SOURCE, indent=2) + '\n')
print('MLX conversion:', MODEL_DIR)

Reusing pinned conversion: /Users/ivywatitwa/medgemma-vet-diagnostics/checkpoints/mlx-openbmb-minicpm-v46-4bit
MLX conversion: /Users/ivywatitwa/medgemma-vet-diagnostics/checkpoints/mlx-openbmb-minicpm-v46-4bit


## Smoke run, then one balanced pass
The exported train JSONL has 4,332 rows (2,166 per class); the first four rows are balanced for the smoke test. LoRA is applied to the language model, the visual input is preserved in each batch, and loss is restricted to assistant completions. Accumulation of four divides the full run evenly: 1,083 optimizer updates. Smoke and full adapters have separate directories.

In [10]:
def train_command(output, extra=()):
    return [sys.executable, '-u', str(TRAINER), '--root', str(ROOT), '--data-dir', str(DATA_DIR), '--model-dir', str(MODEL_DIR), '--output', str(output), '--mode', 'train', '--epochs', '1', '--learning-rate', '2e-4', '--gradient-accumulation-steps', '4', '--max-seq-length', '2048', '--seed', '42', *extra]

if (SMOKE_DIR / 'adapters.safetensors').is_file():
    smoke_summary = json.loads((SMOKE_DIR / 'training_summary.json').read_text())
    assert smoke_summary['model_id'] == MODEL_ID and smoke_summary['prompt'] == PROMPT
    print('Matching smoke adapter already exists:', SMOKE_DIR)
else:
    subprocess.run(train_command(SMOKE_DIR, ['--max-train', '4']), check=True)
assert (SMOKE_DIR / 'adapters.safetensors').is_file()
print('Four-example smoke pass saved:', SMOKE_DIR)

#trainable params: 135.885504 M || all params: 1300.42744 M || trainable%: 10.449%
update=1/1 loss=2.62293
Saved adapter to /Users/ivywatitwa/medgemma-vet-diagnostics/adapters/minicpm-v46-t1-smoke-prompt-v1
Four-example smoke pass saved: /Users/ivywatitwa/medgemma-vet-diagnostics/adapters/minicpm-v46-t1-smoke-prompt-v1


In [11]:
if (OUTPUT_DIR / 'adapters.safetensors').is_file():
    summary = json.loads((OUTPUT_DIR / 'training_summary.json').read_text())
    assert summary['model_id'] == MODEL_ID and summary['prompt'] == PROMPT
    assert summary['train_records'] == 4332 and summary['optimizer_updates'] == 1083, summary
    print('Matching full adapter already exists:', OUTPUT_DIR)
else:
    subprocess.run(train_command(OUTPUT_DIR), check=True)
    summary = json.loads((OUTPUT_DIR / 'training_summary.json').read_text())
assert summary['train_records'] == 4332 and summary['optimizer_updates'] == 1083, summary
assert summary['effective_class_counts'] == {'negative_control': 2166, 'trypanosome_present': 2166}, summary
print(json.dumps(summary, indent=2))

#trainable params: 135.885504 M || all params: 1300.42744 M || trainable%: 10.449%
update=1/1083 loss=2.62293
update=10/1083 loss=0.18015
update=20/1083 loss=0.15606
update=30/1083 loss=0.16699
update=40/1083 loss=0.17904
update=50/1083 loss=0.04467
update=60/1083 loss=0.00862
update=70/1083 loss=0.13785
update=80/1083 loss=0.15013
update=90/1083 loss=0.12917
update=100/1083 loss=0.10521
update=110/1083 loss=0.11133
update=120/1083 loss=1.64876
update=130/1083 loss=0.10515
update=140/1083 loss=0.07437
update=150/1083 loss=0.01246
update=160/1083 loss=0.00357
update=170/1083 loss=0.07532
update=180/1083 loss=0.07817
update=190/1083 loss=0.00143
update=200/1083 loss=0.00609
update=210/1083 loss=0.13337
update=220/1083 loss=0.11756
update=230/1083 loss=0.00102
update=240/1083 loss=0.00012
update=250/1083 loss=0.06181
update=260/1083 loss=0.02040
update=270/1083 loss=0.06738
update=280/1083 loss=0.00102
update=290/1083 loss=0.00061
update=300/1083 loss=0.00122
update=310/1083 loss=0.00635


## Validation
Greedy classification is evaluated on the 488 validation records using the exact training prompt and 4x image mode. This cell does not load `test.jsonl`. Review per-class recall, balanced accuracy and macro-F1 rather than accuracy alone.

In [12]:
subprocess.run([sys.executable, '-u', str(TRAINER), '--root', str(ROOT), '--data-dir', str(DATA_DIR), '--model-dir', str(MODEL_DIR), '--output', str(OUTPUT_DIR), '--mode', 'eval', '--max-val', '488', '--max-new-tokens', '16'], check=True)
metrics = json.loads((OUTPUT_DIR / 't1_metrics.json').read_text())
assert metrics['records'] == 488, metrics
print(json.dumps(metrics, indent=2))
print('MiniCPM-V 4.6 adapter:', OUTPUT_DIR)

evaluated 1/488
evaluated 25/488
evaluated 50/488
evaluated 75/488
evaluated 100/488
evaluated 125/488
evaluated 150/488
evaluated 175/488
evaluated 200/488
evaluated 225/488
evaluated 250/488
evaluated 275/488
evaluated 300/488
evaluated 325/488
evaluated 350/488
evaluated 375/488
evaluated 400/488
evaluated 425/488
evaluated 450/488
evaluated 475/488
{
  "records": 488,
  "accuracy": 1.0,
  "macro_f1": 1.0,
  "labels": [
    "negative_control",
    "trypanosome_present"
  ],
  "confusion_matrix": [
    [
      16,
      0
    ],
    [
      0,
      472
    ]
  ],
  "per_class_recall": {
    "negative_control": 1.0,
    "trypanosome_present": 1.0
  },
  "balanced_accuracy": 1.0,
  "negative_control_recall": 1.0,
  "invalid_outputs": 0,
  "status": "PASS",
  "model_id": "openbmb/MiniCPM-V-4.6",
  "image_mode": "4x"
}
{
  "records": 488,
  "accuracy": 1.0,
  "macro_f1": 1.0,
  "labels": [
    "negative_control",
    "trypanosome_present"
  ],
  "confusion_matrix": [
    [
      16,
   

In [2]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd().resolve()
TRAINER = ROOT / "scripts/train_t1_minicpm_mlx.py"
DATA_DIR = ROOT / "data/secondary/mlx_t1_minicpmv46"
MODEL_DIR = ROOT / "checkpoints/mlx-openbmb-minicpm-v46-4bit"
OUTPUT_DIR = ROOT / "adapters/minicpm-v46-t1-balanced-prompt-v1"
test_path = DATA_DIR / "test.jsonl"

assert TRAINER.is_file(), f"Trainer not found: {TRAINER}"
assert test_path.is_file(), f"Test split not found: {test_path}"
assert (MODEL_DIR / "config.json").is_file(), f"Converted model not found: {MODEL_DIR}"
assert (OUTPUT_DIR / "adapters.safetensors").is_file(), f"Trained adapter not found: {OUTPUT_DIR}"

test_rows = [line for line in test_path.read_text().splitlines() if line.strip()]
assert test_rows, f"No held-out test examples found: {test_path}"

subprocess.run([
    sys.executable, "-u", str(TRAINER),
    "--root", str(ROOT),
    "--data-dir", str(DATA_DIR),
    "--model-dir", str(MODEL_DIR),
    "--output", str(OUTPUT_DIR),
    "--mode", "eval",
    "--split", "test",
    "--max-new-tokens", "16",
], check=True)

test_metrics = json.loads((OUTPUT_DIR / "t1_test_metrics.json").read_text())
assert test_metrics["records"] == len(test_rows), test_metrics
print(json.dumps(test_metrics, indent=2))

evaluated 1/477
evaluated 25/477
evaluated 50/477
evaluated 75/477
evaluated 100/477
evaluated 125/477
evaluated 150/477
evaluated 175/477
evaluated 200/477
evaluated 225/477
evaluated 250/477
evaluated 275/477
evaluated 300/477
evaluated 325/477
evaluated 350/477
evaluated 375/477
evaluated 400/477
evaluated 425/477
evaluated 450/477
evaluated 475/477
{
  "records": 477,
  "accuracy": 0.9769392033542977,
  "macro_f1": 0.5710782310144691,
  "labels": [
    "negative_control",
    "trypanosome_present"
  ],
  "confusion_matrix": [
    [
      1,
      11
    ],
    [
      0,
      465
    ]
  ],
  "per_class_recall": {
    "negative_control": 0.08333333333333333,
    "trypanosome_present": 1.0
  },
  "balanced_accuracy": 0.5416666666666666,
  "negative_control_recall": 0.08333333333333333,
  "invalid_outputs": 0,
  "status": "PASS",
  "model_id": "openbmb/MiniCPM-V-4.6",
  "image_mode": "4x"
}
{
  "records": 477,
  "accuracy": 0.9769392033542977,
  "macro_f1": 0.5710782310144691,
  "la